In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)

In [4]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10),
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [6]:
model=CNN()

criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [7]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        output=model.forward(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss+=loss.item()

    print(f"epochs={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epochs=1/10 & loss=1.361334376246728
epochs=2/10 & loss=0.9245790776694217
epochs=3/10 & loss=0.7466484132172811
epochs=4/10 & loss=0.6229763955563841
epochs=5/10 & loss=0.5169654930644023
epochs=6/10 & loss=0.42232065838392435
epochs=7/10 & loss=0.33866492975169743
epochs=8/10 & loss=0.2627166946754431
epochs=9/10 & loss=0.20202871122399865
epochs=10/10 & loss=0.1562779498717669


In [10]:
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)
        _, predicted=torch.max(outputs,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)

print(f"accuracy={correct_labels/total_labels*100}")


accuracy=74.53
